# Lesson 6.4 — CloudFront, WAF, Shield & Global Accelerator

**Level: Intermediate** | **Exam:** SOA-C03 Domain 5, Task 5.2 (Skill 5.2.3); SAP-C02 Domain 2, Task 2.3

## Learning Objectives
- Configure CloudFront distributions with origin groups and cache behaviors
- Create AWS WAF web ACLs with managed rule groups
- Distinguish Shield Standard vs Advanced
- Design a Global Accelerator architecture for latency-sensitive applications
- Troubleshoot CloudFront caching issues with cache invalidation and cache headers

_Reference: [CloudFront Docs](https://docs.aws.amazon.com/cloudfront/latest/APIReference/) | [AWS WAF](https://docs.aws.amazon.com/waf/latest/developerguide/) | [Global Accelerator](https://docs.aws.amazon.com/global-accelerator/latest/dg/)_

## CloudFront — Content Delivery Network

### How CloudFront Works
```
User (London) → CF Edge Location (London) → Origin (S3/ALB in us-east-1)

If cached at edge:
User → Edge → HIT → Return cached content (fast, low cost)

If not cached:
User → Edge → MISS → Fetch from origin → Cache → Return content
```

### CloudFront Origins
- **S3 Bucket**: Static website hosting, OAC (Origin Access Control) prevents direct S3 access
- **ALB / EC2**: Dynamic content, API responses
- **API Gateway**: Serverless API with CloudFront in front
- **Custom origin**: Any HTTP server (on-prem, other cloud)

### Cache Behaviors
Define caching rules per path pattern:
```
/* (default)           → Cache everything for 1 day
/api/*                 → Never cache (forward headers, cookies)
/images/*              → Cache for 30 days (immutable static assets)
/user/*                → Forward cookies, no caching
```

### Key CloudFront Concepts
- **TTL (Time to Live)**: How long objects are cached at edge. Default: 24 hours.
- **Cache Invalidation**: Force CloudFront to refetch from origin. Costs $0.005/1000 paths. Avoid in hot paths — use versioned URLs instead.
- **OAC (Origin Access Control)**: Restricts S3 access to CloudFront only (replaces OAI)
- **Geo-restriction**: Block/allow content by country

## AWS WAF (Web Application Firewall)

Protects against OWASP Top 10 attacks:
- **SQL Injection** (SQLi)
- **Cross-Site Scripting** (XSS)
- **HTTP flood / DDoS** at Layer 7
- **Bad bots** / scrapers

Integrates with: CloudFront, ALB, API Gateway, AppSync

**WAF Cost:** $1/WebACL/month + $0.60/rule/month + $0.60/1M requests

## AWS Shield
| | Shield Standard | Shield Advanced |
|--|----------------|------------------|
| **Cost** | Free | $3,000/month |
| **Protection** | L3/L4 DDoS (volumetric) | L3/L4/L7 + intelligent detection |
| **SRT access** | No | Yes (24/7 DDoS Response Team) |
| **Cost protection** | No | Yes (credits for scaling costs during attack) |

## AWS Global Accelerator vs CloudFront
| | CloudFront | Global Accelerator |
|--|-----------|--------------------|
| **Caches** | Yes | No |
| **Protocol** | HTTP/HTTPS | TCP, UDP, HTTP |
| **Use case** | Static/cacheable content | Low-latency TCP/UDP, gaming, IoT |
| **Anycast IPs** | No (DNS-based) | Yes (static Anycast IPs) |
| **Health checks** | No | Yes (endpoint health + fast failover) |

In [ ]:
import boto3, json

cf = boto3.client('cloudfront', region_name='us-east-1')
waf = boto3.client('wafv2', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# List CloudFront distributions
try:
    dists = cf.list_distributions()['DistributionList']
    items = dists.get('Items', [])
    if items:
        print(f'CloudFront Distributions ({len(items)}):')
        for d in items:
            domains = [o['DomainName'] for o in d['Origins']['Items']]
            print(f'  {d["Id"]:20} {d["DomainName"]:45} Origins: {", ".join(domains[:2])}')
    else:
        print('No CloudFront distributions found.')
except Exception as e:
    print(f'CloudFront: {e}')

In [ ]:
# Create a WAF Web ACL with AWS Managed Rule Groups
# This protects against common web attacks out-of-the-box

print('WAF WebACL Structure with Managed Rules:')
print()

webacl_config = {
    'Name': 'cloudops-learning-webacl',
    'Scope': 'CLOUDFRONT',  # or 'REGIONAL' for ALB/API GW
    'DefaultAction': {'Allow': {}},  # Default: allow unless a rule blocks
    'Rules': [
        {
            'Name': 'AWS-AWSManagedRulesCommonRuleSet',
            'Priority': 1,
            'OverrideAction': {'None': {}},  # Use rule's action (typically Block)
            'Statement': {
                'ManagedRuleGroupStatement': {
                    'VendorName': 'AWS',
                    'Name': 'AWSManagedRulesCommonRuleSet'  # OWASP Top 10 protection
                }
            },
            'VisibilityConfig': {
                'SampledRequestsEnabled': True,
                'CloudWatchMetricsEnabled': True,
                'MetricName': 'CommonRules'
            }
        },
        {
            'Name': 'AWS-AWSManagedRulesAmazonIpReputationList',
            'Priority': 2,
            'OverrideAction': {'None': {}},
            'Statement': {
                'ManagedRuleGroupStatement': {
                    'VendorName': 'AWS',
                    'Name': 'AWSManagedRulesAmazonIpReputationList'  # Block known bad IPs
                }
            },
            'VisibilityConfig': {
                'SampledRequestsEnabled': True,
                'CloudWatchMetricsEnabled': True,
                'MetricName': 'IpReputation'
            }
        },
        {
            'Name': 'RateLimitRule',
            'Priority': 3,
            'Action': {'Block': {}},
            'Statement': {
                'RateBasedStatement': {
                    'Limit': 1000,           # Max 1000 requests per 5 minutes per IP
                    'AggregateKeyType': 'IP'
                }
            },
            'VisibilityConfig': {
                'SampledRequestsEnabled': True,
                'CloudWatchMetricsEnabled': True,
                'MetricName': 'RateLimit'
            }
        }
    ],
    'VisibilityConfig': {
        'SampledRequestsEnabled': True,
        'CloudWatchMetricsEnabled': True,
        'MetricName': 'cloudops-learning-webacl'
    }
}

print(json.dumps(webacl_config, indent=2))

In [ ]:
# CloudFront Troubleshooting Tips

print('CloudFront Cache Behavior Troubleshooting Guide:')
print()
print('1. Cache MISS when expecting HIT:')
print('   - Check Cache-Control header from origin: must be max-age > 0')
print('   - Check if query strings/cookies are being forwarded (makes cache key unique)')
print('   - Use x-cache: Hit from cloudfront vs Miss from cloudfront response header')
print()
print('2. Stale content after update:')
print('   - Option A: Create invalidation (/*) — costs $0.005/1000 paths')
print('   - Option B: Versioned URLs (preferred): /app-v1.2.js → /app-v1.3.js')
print("   - Option C: Short TTL on frequently changing objects (Cloud-Control: max-age=60)")
print()
print('3. Origin unreachable errors (5xx from CloudFront):')
print('   - Check CloudFront access logs in S3')
print('   - Check origin server health (ALB target group)')
print('   - Configure Origin Group for failover: Primary origin + Secondary origin')
print()
print('4. 403 Forbidden from S3 origin:')
print('   - Use OAC (Origin Access Control) instead of OAI (deprecated)')
print('   - Check S3 bucket policy grants CloudFront service principal GetObject')

## Exercise

In [ ]:
# TODO: Map each scenario to the correct AWS service
# CloudFront / WAF / Shield Advanced / Global Accelerator

scenarios = [
    {'scenario': 'Protect ALB from SQL injection and XSS attacks', 'service': 'TODO'},
    {'scenario': 'Deliver static images from S3 to global users with low latency', 'service': 'TODO'},
    {'scenario': 'Need static anycast IPs for a gaming UDP server with fast failover', 'service': 'TODO'},
    {'scenario': 'Company is being DDoS attacked; need 24/7 response team support', 'service': 'TODO'},
    {'scenario': 'Block all requests from specific countries from accessing the website', 'service': 'TODO'},
]

for s in scenarios:
    print(f"Scenario: {s['scenario']}")
    print(f"  Service: {s['service']}")
    print()

## Summary
- **CloudFront**: CDN with 400+ edge locations; caches HTTP/HTTPS content globally; integrates with WAF
- **Cache behaviors**: Define per path pattern; `/api/*` = no cache; `/static/*` = long cache
- **WAF**: Attach to CloudFront, ALB, or API Gateway; use managed rule groups for OWASP Top 10
- **Shield Standard**: Free, protects all AWS customers from L3/L4 volumetric DDoS
- **Shield Advanced**: $3k/mo; adds L7 protection, DDoS Response Team, cost protection
- **Global Accelerator**: Static Anycast IPs + intelligent routing over AWS backbone; no caching; for TCP/UDP

---
**Next Phase:** [Phase 7 — Organizational Complexity & Multi-Account](../phase-7-org-complexity/lesson-7.1-multi-account.ipynb)